In [ ]:
#sunburst for chapter 3 FLB quantification
#byproduct across three regions combine with quantity in bar chart

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from matplotlib.patches import Wedge
from matplotlib.ticker import FuncFormatter

# Shared publication style used by the other project figures.
TEXT_COLOR = "#303438"
GRID_COLOR = "#D9DEE1"
REGION_COLORS = {
    "China": "#245B66",
    "The U.S.": "#A86F58",
    "The EU": "#81997B",
}
LABEL_THRESHOLD_PERCENT = 0.5

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 14,
    "text.color": TEXT_COLOR,
    "axes.labelcolor": TEXT_COLOR,
})

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "figures").is_dir())
input_file = repo_root / "figures" / "input_data" / "FLB_Qty_Sunburst.xlsx"
df = pd.read_excel(input_file, sheet_name="Sheet1")
required = {"region", "byproduct", "quantity", "percent_of_total"}
missing = required.difference(df.columns)
if missing:
    raise ValueError(f"{input_file.name}: missing columns {sorted(missing)}")

df = df[list(required)].dropna(subset=["region", "byproduct", "quantity", "percent_of_total"]).copy()
df["quantity"] = pd.to_numeric(df["quantity"], errors="raise")
df["percent_of_total"] = pd.to_numeric(df["percent_of_total"], errors="raise")
df["region"] = (
    df["region"].astype(str).str.strip()
    .replace({"TheU.S.": "The U.S.", "The U.S.": "The U.S.", "The EU": "The EU"})
)
df["byproduct"] = df["byproduct"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

region_order = ["China", "The U.S.", "The EU"]
df["region"] = pd.Categorical(df["region"], categories=region_order, ordered=True)
df = df.sort_values(["region"], kind="stable").reset_index(drop=True)
df.head()


In [ ]:
def create_radial_sunburst_chart(data_df, output_stem="sunburst_ch3_quantification_optimized"):
    """Create a publication-style sunburst plus radial quantity bars."""
    data = data_df.copy()
    region_totals = data.groupby("region", observed=True)["percent_of_total"].sum()
    total_percent = data["percent_of_total"].sum()
    max_quantity = float(data["quantity"].max())

    # Geometry leaves room for readable radial labels and a right-side legend.
    inner_radius = 0.18
    region_radius = 0.43
    product_radius = 0.66
    bar_start_radius = 0.76
    max_bar_height = 0.55
    label_offset = 0.035  # Keep each label close to its own radial-bar tip.

    fig, ax = plt.subplots(figsize=(21, 21), facecolor="white")
    ax.set_aspect("equal")
    ax.set_xlim(-1.88, 2.72)
    ax.set_ylim(-1.88, 1.88)
    ax.axis("off")

    # Draw shared circular quantity scale behind the radial bars.
    tick_step = 10_000
    scale_max = np.ceil(max_quantity / tick_step) * tick_step
    tick_values = np.arange(tick_step, scale_max + tick_step, tick_step)
    tick_angle = np.radians(72)
    for tick in tick_values:
        radius = bar_start_radius + (tick / scale_max) * max_bar_height
        ax.add_patch(plt.Circle(
            (0, 0), radius, fill=False, color=GRID_COLOR,
            linestyle=(0, (3, 3)), linewidth=1.0, zorder=0,
        ))
        x, y = radius * np.cos(tick_angle), radius * np.sin(tick_angle)
        ax.text(
            x + 0.025, y, f"{tick / 1000:.0f}k",
            ha="left", va="center", fontsize=12, color="#59636A",
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.82, "pad": 1.5},
            zorder=7,
        )
    ax.text(
        1.43 * np.cos(tick_angle), 1.43 * np.sin(tick_angle),
        r"FLB quantity (kt wet mass yr$^{-1}$)",
        ha="left", va="center", fontsize=13, fontweight="semibold",
        color=TEXT_COLOR,
    )

    start_angle = 90.0
    for region in region_order:
        region_data = data[data["region"] == region]
        if region_data.empty:
            continue
        region_percent = float(region_totals.loc[region])
        region_angle = 360.0 * region_percent / total_percent
        region_color = REGION_COLORS[region]

        # Inner region sector.
        ax.add_patch(Wedge(
            (0, 0), region_radius, start_angle, start_angle + region_angle,
            width=region_radius - inner_radius,
            facecolor=region_color, alpha=0.95,
            edgecolor="white", linewidth=2.2, zorder=2,
        ))
        region_mid = np.radians(start_angle + region_angle / 2)
        region_label_radius = (inner_radius + region_radius) / 2
        ax.text(
            region_label_radius * np.cos(region_mid),
            region_label_radius * np.sin(region_mid),
            region, ha="center", va="center", fontsize=14,
            fontweight="semibold", color="white", zorder=6,
        )

        # Product sectors and quantity bars.
        product_start = start_angle
        for row in region_data.itertuples(index=False):
            product_angle = 360.0 * float(row.percent_of_total) / total_percent
            product_end = product_start + product_angle
            mid_angle_deg = product_start + product_angle / 2
            mid_angle = np.radians(mid_angle_deg)

            ax.add_patch(Wedge(
                (0, 0), product_radius, product_start, product_end,
                width=product_radius - region_radius,
                facecolor=region_color, alpha=0.58,
                edgecolor="white", linewidth=1.2, zorder=2,
            ))

            bar_height = (float(row.quantity) / scale_max) * max_bar_height
            gap = min(product_angle * 0.12, 0.7)
            ax.add_patch(Wedge(
                (0, 0), bar_start_radius + bar_height,
                product_start + gap, product_end - gap,
                width=bar_height, facecolor=region_color, alpha=0.90,
                edgecolor="white", linewidth=0.75, zorder=3,
            ))

            if float(row.percent_of_total) >= LABEL_THRESHOLD_PERCENT:
                label_radius = bar_start_radius + bar_height + label_offset
                x, y = label_radius * np.cos(mid_angle), label_radius * np.sin(mid_angle)
                if 90 < mid_angle_deg % 360 < 270:
                    rotation = mid_angle_deg + 180
                    alignment = "right"
                else:
                    rotation = mid_angle_deg
                    alignment = "left"
                ax.text(
                    x, y, str(row.byproduct),
                    ha=alignment, va="center", rotation=rotation,
                    rotation_mode="anchor", fontsize=16,
                    fontweight="semibold", color=TEXT_COLOR,
                    bbox={"boxstyle": "round,pad=0.18", "facecolor": "white", "edgecolor": "none", "alpha": 0.78},
                    zorder=8,
                )
            product_start = product_end
        start_angle += region_angle

    ax.add_patch(plt.Circle(
        (0, 0), inner_radius, facecolor="white",
        edgecolor="#697277", linewidth=1.0, zorder=5,
    ))
    ax.text(
        0, 0, "Food\nbyproduct", ha="center", va="center",
        fontsize=14.5, linespacing=1.05, fontweight="semibold", color=TEXT_COLOR, zorder=6,
    )

    legend_handles = [
        mpatches.Patch(facecolor=REGION_COLORS[region], edgecolor="none", label=region)
        for region in region_order
    ]
    legend = ax.legend(
        handles=legend_handles, title="Region", loc="center left",
        bbox_to_anchor=(0.83, 0.50), frameon=False,
        fontsize=14, title_fontsize=15, labelspacing=0.9,
    )
    legend.get_title().set_fontweight("semibold")

    ax.text(
        0.99, 0.035,
        f"Byproduct labels shown for shares >= {LABEL_THRESHOLD_PERCENT:.1f}%",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=12, color="#626B70",
    )

    fig.tight_layout()
    output_stem = Path(output_stem)
    fig.savefig(output_stem.with_suffix(".png"), dpi=600, bbox_inches="tight", facecolor="white")
    fig.savefig(output_stem.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
    fig.savefig(output_stem.with_suffix(".svg"), bbox_inches="tight", facecolor="white")
    return fig, ax


output_stem = repo_root / "figures" / "main" / "sunburst_ch3_quantification_optimized"
output_stem.parent.mkdir(parents=True, exist_ok=True)
fig, ax = create_radial_sunburst_chart(df, output_stem)
plt.show()

region_summary = (
    df.groupby("region", observed=True)
    .agg(quantity=("quantity", "sum"), share_percent=("percent_of_total", "sum"))
)
region_summary
